In [ ]:
# =============================================
# CELL 1: Setup (run once per session)
#   Phase 0A GPU sweep — Tesla T4
#   Requires a GPU accelerator: Settings → Accelerator → GPU T4 x2
# =============================================
import sys, subprocess, os

# --- Secrets: loaded from Kaggle Secrets, never hardcoded ---
#   Add in Kaggle: Add-ons → Secrets → KAGGLE_USERNAME, KAGGLE_KEY,
#   NVIDIA_API_KEY (optional: NVIDIA_API_KEY_1, GROQ_API_KEY)
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "NVIDIA_API_KEY",
                "NVIDIA_API_KEY_1", "GROQ_API_KEY"):
        try:
            _v = _s.get_secret(_k)
            if _v:
                os.environ[_k] = _v
                print(f"[OK] secret {_k} loaded")
        except Exception:
            pass
except Exception as e:
    print(f"WARN secrets unavailable: {e}")

# --- Install the audited GPU stack ---
#   PINNED ON PURPOSE — do NOT loosen to qiskit>=1.0 / qiskit-aer>=0.14:
#   Aer 0.15.1 needs qiskit.providers.convert_to_target (removed in Qiskit 2.0)
#   and qft.py uses the QFT class (removed in Qiskit 2.1). A loose range pulls
#   Qiskit 2.x and breaks the import. qiskit-aer-gpu==0.15.1 is the GPU build
#   verified working on Kaggle T4 CUDA.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "qiskit", "qiskit-aer", "qiskit-aer-gpu", "qiskit-terra"], check=False)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "qiskit==1.4.2", "qiskit-aer-gpu==0.15.1",
    "pynvml", "nvidia-ml-py", "pyyaml",
    "pandas", "kaggle>=2.2.2", "requests",
    "python-dotenv", "networkx",
])
os.environ["OPENQSIM_DEVICE"] = "GPU"

# --- Clone / update repo ---
REPO_DIR = "/kaggle/working/opensim-ai"
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone",
                           "https://github.com/lekkalaharsha/opensim-ai", REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", "main"])
sys.path.insert(0, REPO_DIR)

print("[OK] Setup complete (qiskit==1.4.2 + qiskit-aer-gpu==0.15.1, device=GPU)")

In [ ]:
# =============================================
# CELL 2: Run the GPU statevector sweep
#   Run CELL 1 first (same session).
# =============================================
import os, sys, subprocess, shutil, importlib.util, re, json
from pathlib import Path

REPO_DIR        = "/kaggle/working/opensim-ai"
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "harshalekkala1")
KAGGLE_DATASET  = f"{KAGGLE_USERNAME}/openqsim-phase0a-gpu"
OUTPUT_DIR      = "/kaggle/working/benchmark_outputs"
SWEEP_CONFIG    = f"{REPO_DIR}/benchmark/sweep_config_0a_gpu.yaml"
KAGGLE_MOD_DIR  = f"{REPO_DIR}/kaggle"
if os.environ.get("NVIDIA_API_KEY_1"):
    os.environ["NVIDIA_API_KEY_COUNT"] = "2"
    print("[OK] Dual NVIDIA key mode")

# --- Patch package-relative imports for flat sys.path loading ---
def _patch(path):
    txt = open(path).read()
    txt = re.sub(r'from kaggle\.(\w+) import', r'from \1 import', txt)
    txt = re.sub(r'import kaggle\.(\w+)', r'import \1', txt)
    open(path, "w").write(txt)

for py in Path(KAGGLE_MOD_DIR).glob("*.py"):
    _patch(str(py))
_runner_py = f"{REPO_DIR}/benchmark/runner.py"
if os.path.exists(_runner_py):
    _patch(_runner_py)
sys.path.insert(0, KAGGLE_MOD_DIR)

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

KaggleRunner     = _load("kaggle_runner",    f"{KAGGLE_MOD_DIR}/runner.py").KaggleRunner
DatasetAssembler = _load("kaggle_assembler", f"{KAGGLE_MOD_DIR}/dataset_assembler.py").DatasetAssembler
KaggleAPIClient  = _load("kaggle_client",    f"{KAGGLE_MOD_DIR}/api_client.py").KaggleAPIClient

# --- Verify Aer actually exposes a usable GPU (abort if not) ---
from backend.environment import collect_environment_metadata
env = collect_environment_metadata()
print(f"GPU: {env.gpu_name} ({env.gpu_memory_mb} MB) | Qiskit: {env.qiskit_version}")
from qiskit_aer import AerSimulator
devices = AerSimulator().available_devices()
print(f"Aer devices: {devices}")
if "GPU" not in devices:
    raise RuntimeError(
        "Aer does not expose a GPU. Enable a GPU accelerator in Kaggle and "
        "re-run CELL 1. Aborting to avoid a silent CPU run mislabeled as GPU."
    )
print("[OK] GPU confirmed — starting sweep")

if KaggleAPIClient(dataset=KAGGLE_DATASET).dataset_exists():
    print(f"[OK] Dataset exists: {KAGGLE_DATASET}")
else:
    print(f"WARN Dataset not found: {KAGGLE_DATASET} — will zip as fallback")

# --- Run sweep (360 combos) ---
runner = KaggleRunner(
    sweep_config_path=SWEEP_CONFIG,
    output_dir=OUTPUT_DIR,
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset=KAGGLE_DATASET,
    use_advisor=True,
)
sweep_result = runner.run()
print(f"\nSweep done: {sweep_result.completed_count}/{sweep_result.total_combinations}")
print(f"OOM: {sweep_result.oom_count}  Errors: {sweep_result.error_count}")
print(f"Time: {sweep_result.duration_seconds/3600:.1f}h")

# --- Assemble + push ---
assembler = DatasetAssembler(
    raw_dir=f"{OUTPUT_DIR}/raw",
    dataset_dir=f"{OUTPUT_DIR}/datasets/openqsim_v0.1-gpu",
)
manifest = assembler.assemble()
print(f"{manifest.records} records | {manifest.successful_runs} successful")

dataset_dir = Path(OUTPUT_DIR)
(dataset_dir / "dataset-metadata.json").write_text(json.dumps({
    "title": "OpenQSim Phase 0A GPU Benchmark Outputs",
    "id": KAGGLE_DATASET,
    "licenses": [{"name": "MIT"}],
}))
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(dataset_dir),
     "-m", f"Phase 0A GPU complete: {manifest.records} records"],
    capture_output=True, text=True,
)
if r.returncode == 0:
    print(f"[OK] Pushed to {KAGGLE_DATASET}")
else:
    zip_path = shutil.make_archive(
        "/kaggle/working/benchmark_outputs_gpu_backup", "zip", str(OUTPUT_DIR))
    print(f"WARN Push failed: {r.stderr[:200]}")
    print(f"[OK] Zip saved: {zip_path} — download from Output tab")

print("\n" + "=" * 50)
print("OPENQSIM PHASE 0A GPU SWEEP COMPLETE")
print(f"  Records:   {manifest.records}")
print(f"  Successes: {manifest.successful_runs}")
print(f"  Backends:  {manifest.backends}")
print(f"  Qubits:    {manifest.qubit_range}")
print(f"  Dataset:   {KAGGLE_DATASET}")
print("=" * 50)